In [1]:
# ── Imports ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ta  
# ta = Technical Analysis
# contient tous les indicateurs financiers 

plt.style.use("dark_background")

# ── Chargement des données sauvegardées ───────────────────────
df = pd.read_csv("../data/btc_raw.csv", index_col=0, parse_dates=True)
# index_col=0   = la première colonne (les dates) devient l'index
# parse_dates=True = convertit automatiquement les dates en format datetime

print(f" {len(df)} lignes chargées")
df.head()

 1824 lignes chargées


,Open,High,Low,Close,Volume,returns,log_returns
Date,,,,,,,
2020-01-03 00:00:00+00:00,6984.428711,7413.715332,6914.996094,7344.884277,28111481032,0.051452,0.050172
2020-01-04 00:00:00+00:00,7345.375488,7427.385742,7309.514160,7410.656738,18444271275,0.008955,0.008915
2020-01-05 00:00:00+00:00,7410.451660,7544.497070,7400.535645,7411.317383,19725074095,0.000089,0.000089
2020-01-06 00:00:00+00:00,7410.452148,7781.867188,7409.292969,7769.219238,23276261598,0.048291,0.047161
2020-01-07 00:00:00+00:00,7768.682129,8178.215820,7768.227539,8163.692383,28767291327,0.050774,0.049527


# RSI — Relative Strength Index

Indicateur entre 0 et 100 qui mesure si le BTC est suracheté ou survendu :

- RSI > 70 → suracheté → potentiel retournement à la baisse
- RSI < 30 → survendu → potentiel rebond à la hausse
- RSI = 50 → neutre

## Comment est-il calculé ?

1. Séparer les jours de hausse et de baisse
2. Faire la moyenne des hausses et baisses en divisant toujours par 14
    - 7 jours  → trop sensible, trop de faux signaux
    - 14 jours → bon équilibre signal / stabilité ✅
    - 28 jours → trop lent, RSI réagit trop tard
3. Calculer RS puis RSI
    - RS  = Moyenne hausses / Moyenne baisses
    - RSI = 100 - (100 / (1 + RS))

In [2]:
# ── RSI (Relative Strength Index) ─────────────────────────────

df["rsi"] = ta.momentum.RSIIndicator(
    close=df["Close"],  # on l'applique sur le prix de clôture
    window=14           # convention standard : on regarde les 14 derniers jours
                        # 14 jours = compromis entre sensibilité et stabilité
).rsi()
# la librairie ta calcule tout automatiquement pour nous

print("── Statistiques du RSI ──")
print(f"RSI moyen : {df['rsi'].mean():.2f}")
print(f"RSI max   : {df['rsi'].max():.2f}")
print(f"RSI min   : {df['rsi'].min():.2f}")

df[["Close", "rsi"]].tail(10)
#affichage des dernieres lignes

── Statistiques du RSI ──
RSI moyen : 53.61
RSI max   : 90.72
RSI min   : 14.00


,Close,rsi
Date,,
2024-12-21 00:00:00+00:00,97224.726562,48.952697
2024-12-22 00:00:00+00:00,95104.937500,44.934433
2024-12-23 00:00:00+00:00,94686.242188,44.163328
2024-12-24 00:00:00+00:00,98676.093750,52.524176
2024-12-25 00:00:00+00:00,99299.195312,53.690417
2024-12-26 00:00:00+00:00,95795.515625,46.737993
2024-12-27 00:00:00+00:00,94164.859375,43.889453
2024-12-28 00:00:00+00:00,95163.929688,46.058619
2024-12-29 00:00:00+00:00,93530.226562,43.122878


RSI moyen : 53.61  → légèrement au dessus de 50, cohérent avec la tendance 
                     haussière globale du BTC sur 5 ans 

RSI max   : 90.72  → bull run extrême (probablement fin 2020 ou fin 2024)

RSI min   : 14.00  → crash extrême (probablement mars 2020 COVID)

## Volatilité Glissante (Rolling Volatility)

Ecart-type des rendements calculé sur une fenêtre mobile de jours.

### Pourquoi glissante ?
- La volatilité globale (3.36%) est une moyenne sur 5 ans → pas dynamique
- La volatilité glissante évolue dans le temp* → bien plus utile pour le ML

### Fenêtres utilisées
- 7 jours  → agitation sur la dernière semaine (court terme)
- 30 jours → agitation sur le dernier mois (moyen terme)

### Fonctionnement
```
Jour 8  → écart-type des jours 1 à 7
Jour 9  → écart-type des jours 2 à 8
Jour 10 → écart-type des jours 3 à 9
```
 Les premières lignes = NaN car pas assez de jours pour remplir la fenêtre


In [3]:
# ── Volatilité Glissante ───────────────────────────────────────

df["volatility_7"]  = df["returns"].rolling(window=7).std()
# rolling(window=7) = fenêtre glissante de 7 jours (une semaine en arriere)
# .std()            = écart-type sur cette fenêtre

df["volatility_30"] = df["returns"].rolling(window=30).std()
# même chose sur 30 jours(le dernier mois)


print("── Volatilité moyenne par fenêtre ──")
print(f"Volatilité 7j  : {df['volatility_7'].mean()*100:.2f}%")
print(f"Volatilité 30j : {df['volatility_30'].mean()*100:.2f}%")

df[["Close", "returns", "volatility_7", "volatility_30"]].tail(10)

── Volatilité moyenne par fenêtre ──
Volatilité 7j  : 2.93%
Volatilité 30j : 3.12%


,Close,returns,volatility_7,volatility_30
Date,,,,
2024-12-21 00:00:00+00:00,97224.726562,-0.005434,0.028516,0.024357
2024-12-22 00:00:00+00:00,95104.937500,-0.021803,0.024454,0.024652
2024-12-23 00:00:00+00:00,94686.242188,-0.004402,0.021330,0.024569
2024-12-24 00:00:00+00:00,98676.093750,0.042138,0.030453,0.025787
2024-12-25 00:00:00+00:00,99299.195312,0.006315,0.022331,0.023958
2024-12-26 00:00:00+00:00,95795.515625,-0.035284,0.024347,0.024803
2024-12-27 00:00:00+00:00,94164.859375,-0.017022,0.024814,0.023737
2024-12-28 00:00:00+00:00,95163.929688,0.010610,0.025506,0.023813
2024-12-29 00:00:00+00:00,93530.226562,-0.017167,0.024985,0.023742


Volatilité 7j  : 2.93%  → agitation sur 1 semaine  → plus faible

Volatilité 30j : 3.12%  → agitation sur 1 mois     → légèrement plus élevée

explication : sur un mois on a plus de hausses et baisses que sur une semaien , ce qui explique la difference de volatilité

## Les Lags (Mémoire du passé)

Un lag = valeur d'une colonne décalée de N jours dans le passé.

### Pourquoi ?
Le modèle ML n'a pas de mémoire naturelle — les lags lui donnent
le contexte des jours précédents pour mieux prédire demain.

### Lags utilisés
- **lag_1** → prix de clôture d'hier      (J-1)
- **lag_2** → prix de clôture avant-hier  (J-2)
- **lag_5** → prix de clôture il y a 5j   (J-5)

### Fonctionnement
```
Date        Close    lag_1    lag_2    lag_5
─────────────────────────────────────────────
2020-01-06  8000$    7500$    7200$    6800$
2020-01-07  7800$    8000$    7500$    7000$
```
es premières lignes = NaN car pas de passé disponible



In [5]:
# ── Lags (mémoire du passé) ────────────────────────────────────

df["lag_1"] = df["Close"].shift(1)
# shift(1) = prix de clôture d'hier

df["lag_2"] = df["Close"].shift(2)
# prix d'avant-hier

df["lag_5"] = df["Close"].shift(5)
# prix il y a 5 jours (semaine dernière)

# ── Nettoyage final ────────────────────────────────────────────
df.dropna(inplace=True)
# supprime toutes les lignes avec NaN
# (générées par rolling() et shift())

print(f" DataFrame final : {df.shape[0]} lignes | {df.shape[1]} colonnes")
print("\n── Colonnes disponibles ──")
print(df.columns.tolist())

df.tail(5)

 DataFrame final : 1790 lignes | 13 colonnes

── Colonnes disponibles ──
['Open', 'High', 'Low', 'Close', 'Volume', 'returns', 'log_returns', 'rsi', 'volatility_7', 'volatility_30', 'lag_1', 'lag_2', 'lag_5']


,Open,High,Low,Close,Volume,returns,log_returns,rsi,volatility_7,volatility_30,lag_1,lag_2,lag_5
Date,,,,,,,,,,,,,
2024-12-26 00:00:00+00:00,99297.695312,99884.570312,95137.882812,95795.515625,47054980873,-0.035284,-0.035922,46.737993,0.024347,0.024803,99299.195312,98676.093750,97224.726562
2024-12-27 00:00:00+00:00,95704.976562,97294.843750,93310.742188,94164.859375,52419934565,-0.017022,-0.017169,43.889453,0.024814,0.023737,95795.515625,99299.195312,95104.937500
2024-12-28 00:00:00+00:00,94160.187500,95525.898438,94014.289062,95163.929688,24107436185,0.010610,0.010554,46.058619,0.025506,0.023813,94164.859375,95795.515625,94686.242188
2024-12-29 00:00:00+00:00,95174.054688,95174.875000,92881.789062,93530.226562,29635885267,-0.017167,-0.017316,43.122878,0.024985,0.023742,95163.929688,94164.859375,98676.093750
2024-12-30 00:00:00+00:00,93527.195312,94903.320312,91317.132812,92643.210938,56188003691,-0.009484,-0.009529,41.573470,0.025136,0.023730,93530.226562,95163.929688,99299.195312


## Variable Cible (Target)

### Définition
La variable cible `y` est ce que le modèle doit prédire.

### Notre question
Le prix de demain sera-t-il supérieur à celui d'aujourd'hui ?
- **1** → OUI → signal d'achat
- **0** → NON → signal de vente / attente

### Formule
```
target = 1  si  Close(J+1) > Close(J)
target = 0  si  Close(J+1) < Close(J)
```

### Pourquoi classification et pas régression ?
- Régression → prédire le prix exact (ex: 45 231$) → très difficile
- Classification → prédire la direction (monte/descend) → plus réaliste
- En trading, connaître la DIRECTION suffit pour prendre une décision

In [6]:
# ── Variable Cible ─────────────────────────────────────────────

df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
# shift(-1)   = prix de DEMAIN (décalage vers le haut cette fois)
# > df["Close"] = est-ce que demain > aujourd'hui ?
# .astype(int) = convertit True/False en 1/0

# La dernière ligne sera NaN (pas de lendemain)
df.dropna(inplace=True)

print("── Distribution de la cible ──")
print(df["target"].value_counts())
print(f"\nHausse : {df['target'].mean()*100:.1f}% des jours")
print(f"Baisse : {(1-df['target'].mean())*100:.1f}% des jours")


# ── Sauvegarde ─────────────────────────────────────────────────
df.to_csv("../data/btc_features.csv")
print(" Features sauvegardées dans data/btc_features.csv")

── Distribution de la cible ──
target
1    915
0    875
Name: count, dtype: int64

Hausse : 51.1% des jours
Baisse : 48.9% des jours


51.1% de hausses → si on prédit TOUJOURS "hausse"

 on obtient 51.1% de précision

 le modèle XGBoost doit faire MIEUX que 51.1%